# Infrastructure Optimization Engine
## Multi-Objective Optimization Analysis

**Student Name:** Pongsupa Supapa \
**Target Company/Industry:** Cloudflare, Inc. Cloud infrastructure and edge computing \
**Problem:** Find a workload placement and capacity configuration that balances infrastructure cost and response time. \
**Objectives:** Minimize estimated monthly infrastructure cost (USD) vs. minimize predicted demand-weighted response time (ms).

## Project Overview

This project aims to optimize the placement of application workloads across different geographic locations while minimizing infrastructure costs and application response times. This tradeoff is important to Cloudflare because increasing computing capacity and placing applications closer to users can improve performance but also increase operating costs. The decision variables I will control are the geographic placement of workloads and the amount of computing capacity allocated to each location. Unlike other optimization problems where higher objective values are preferred, this project focuses on minimizing both objectives to find the most efficient balance between cost and performance.

## Section 1: Data Acquisition

I use the processed data from my download data notebook, so `download_data.ipynb` needs to be run first. The data is downloaded and prepared separately so that this notebook can focus on the optimization and analysis. The datasets and their relevant variables are documented in `data/DATA_SOURCE.md`.

In [1]:
from pathlib import Path
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display


data_folder = Path("data")
image_folder = Path("images")
image_folder.mkdir(exist_ok=True)

# Load the files prepared by the download notebook.
files = {
    "Alibaba workloads": "alibaba_joined.csv",
    "RIPE ping measurements": "original_ripe_ping_sample.csv",
    "RIPE regional latency": "ripe_region_rtt.csv",
    "Cloudflare pricing": "original_cloudflare_pricing.csv",
    "Cloudflare instances": "original_cloudflare_instances.csv"
}

for filename in files.values():
    if not (data_folder / filename).is_file():
        raise FileNotFoundError(f"Missing {filename}. Run the download data notebook first.")

joined = pd.read_csv(data_folder / files["Alibaba workloads"])
network = pd.read_csv(data_folder / files["RIPE ping measurements"])
network_summary = pd.read_csv(data_folder / files["RIPE regional latency"])
pricing = pd.read_csv(data_folder / files["Cloudflare pricing"])
instances = pd.read_csv(data_folder / files["Cloudflare instances"])

print("Data loaded successfully!")

for name, table in [("Alibaba workloads", joined), ("RIPE ping measurements", network),
                    ("RIPE regional latency", network_summary), ("Cloudflare pricing", pricing),
                    ("Cloudflare instances", instances)]:
    print(f"\n{name}: {table.shape}")
    display(table.head())

Data loaded successfully!

Alibaba workloads: (11959, 8)


,timestamp,msname,msinstanceid,nodeid,providerrpc_rt,providerrpc_mcr,cpu_utilization,memory_utilization
0,0,MS_10025,MS_10025_POD_52,NODE_29161,124.514563,3.034569e-06,0.120409,0.657554
1,0,MS_10025,MS_10025_POD_77,NODE_27651,145.019802,2.975646e-06,0.148975,0.674028
2,0,MS_10026,MS_10026_POD_11,NODE_26482,3.772036,6.461963e-06,0.037432,0.436981
3,0,MS_10042,MS_10042_POD_9,NODE_22628,584.800000,4.910306e-08,0.029340,0.483108
4,0,MS_10077,MS_10077_POD_17,NODE_40145,3.257154,1.211373e-04,0.198450,0.634625



RIPE ping measurements: (95974, 7)


,source_region,target_region,probe_id,target_anchor_id,measurement_id,timestamp,rtt_ms
0,Tokyo,US East,6370,1818,22252930,1794547059,154.250422
1,Tokyo,US East,6370,1818,22252930,1794547059,153.841995
2,Tokyo,US East,6370,1818,22252930,1794547059,155.003348
3,US West,US East,6411,1818,22252930,1794547055,72.849959
4,US West,US East,6411,1818,22252930,1794547055,70.912308



RIPE regional latency: (25, 4)


,source_region,target_region,latency_ms,packet_count
0,London,London,1.854787,2805
1,London,Singapore,164.665832,2798
2,London,Tokyo,230.014783,3881
3,London,US East,70.848953,3853
4,London,US West,141.366336,2786



Cloudflare pricing: (3, 5)


,parameter,rate,unit,source,accessed
0,cpu_vcpu_second,2.000000e-05,USD/vCPU-second,https://developers.cloudflare.com/containers/p...,2026-11-14
1,memory_gib_second,2.500000e-06,USD/GiB-second,https://developers.cloudflare.com/containers/p...,2026-11-14
2,disk_gb_second,7.000000e-08,USD/GB-second,https://developers.cloudflare.com/containers/p...,2026-11-14



Cloudflare instances: (6, 6)


,instance_type,vcpu,memory_gib,disk_gb,source,accessed
0,lite,0.0625,0.25,2.0,https://developers.cloudflare.com/containers/p...,2026-11-14
1,basic,0.2500,1.00,4.0,https://developers.cloudflare.com/containers/p...,2026-11-14
2,standard-1,0.5000,4.00,8.0,https://developers.cloudflare.com/containers/p...,2026-11-14
3,standard-2,1.0000,6.00,12.0,https://developers.cloudflare.com/containers/p...,2026-11-14
4,standard-3,2.0000,8.00,16.0,https://developers.cloudflare.com/containers/p...,2026-11-14


### Check the Processed Data

Before building the deployment configurations, I check that every region has a latency value, there are enough services, and the pricing data has all the rates and container sizes I need.

In [2]:
# Check the processed data before creating deployment configurations.
regions = ["US East", "US West", "London", "Singapore", "Tokyo"]

latency_matrix = network_summary.pivot(
    index="source_region", columns="target_region", values="latency_ms"
)
latency_matrix = latency_matrix.reindex(index=regions, columns=regions)

if latency_matrix.isna().any().any():
    raise ValueError("The processed RIPE data is missing one or more regional routes.")

if joined["msname"].nunique() < 6:
    raise ValueError("At least six distinct services are needed for the model.")

assert pricing["parameter"].is_unique and (pricing["rate"] > 0).all()
assert instances["instance_type"].is_unique

print("Regional latency matrix (ms):")
display(latency_matrix.round(2))

required_columns = {
    "Alibaba workloads": (joined, ["msname", "msinstanceid", "cpu_utilization", "memory_utilization", "providerrpc_rt", "providerrpc_mcr"]),
    "Cloudflare pricing": (pricing, ["parameter", "rate"]),
    "Cloudflare instances": (instances, ["instance_type", "vcpu", "memory_gib", "disk_gb"])
}

for name, (table, columns) in required_columns.items():
    if table.empty or not set(columns).issubset(table.columns):
        raise ValueError(f"Missing required data in {name}.")
    if table[columns].isna().any().any():
        raise ValueError(f"Missing values in {name}.")

if not np.isfinite(latency_matrix.to_numpy()).all() or (latency_matrix <= 0).any().any():
    raise ValueError("RIPE latency values must be finite and positive.")

required_rates = {"cpu_vcpu_second", "memory_gib_second", "disk_gb_second"}
required_sizes = {"basic", "standard-1", "standard-2", "standard-3", "standard-4"}

if not required_rates.issubset(pricing["parameter"]) or not required_sizes.issubset(instances["instance_type"]):
    raise ValueError("The Cloudflare pricing or instance data is incomplete.")

Regional latency matrix (ms):


target_region,US East,US West,London,Singapore,Tokyo
source_region,,,,,
US East,2.38,71.06,69.02,214.80,166.70
US West,68.07,11.97,136.75,173.40,114.47
London,70.85,141.37,1.85,164.67,230.01
Singapore,223.30,172.34,157.58,1.05,76.38
Tokyo,157.32,107.89,232.84,69.99,1.93


## Create Deployment Configurations

I select six distinct services with the most matching measurements and use one observed container per service. Since Alibaba normalizes its resource measurements, I use the published basic container size as a reference to convert each utilization value into estimated CPU and RAM demand. This is an assumption, not a recovery of Alibaba's physical capacity.

I compare all nonempty combinations of five regions and four published capacity tiers. The selected workloads are replicated in each active region, with each client region assigned to its lowest-RTT location. I split demand equally across client regions because the trace has no user geography, and weight the service response times by their normalized call rates as a relative demand proxy. The response-time estimate adds measured geographic RTT to the Alibaba RPC baseline; the original RPC time may already include network delay, so this is only an approximation.

I use peak CPU and memory requirements to check capacity, and mean CPU demand to estimate usage charges. Cost assumes 30 days of continuous activity, provisioned memory and disk, and active CPU usage, before allowances and other charges.

In [3]:
# Build workload profiles from real matched records.
selected_services = joined.groupby("msname").size().sort_values(ascending=False).head(6).index
sample = joined[joined["msname"].isin(selected_services)].copy()

# Use one representative container per service (most matched timestamps).
container_counts = sample.groupby(["msname", "msinstanceid"]).size().reset_index(name="count")
representatives = container_counts.sort_values("count", ascending=False).drop_duplicates("msname")
sample = sample.merge(representatives[["msname", "msinstanceid"]], on=["msname", "msinstanceid"])

sizes = instances.set_index("instance_type")
reference = sizes.loc["basic"]
workload_rows = []

for service, records in sample.groupby("msname"):
    weight = records["providerrpc_mcr"]
    workload_rows.append({
        "service": service,
        "observations": len(records),
        "relative_demand": weight.mean(),
        "rpc_ms": np.average(records["providerrpc_rt"], weights=weight),
        "mean_cpu": records["cpu_utilization"].mean() * reference["vcpu"],
        "peak_cpu": records["cpu_utilization"].max() * reference["vcpu"],
        "peak_memory_gib": records["memory_utilization"].max() * reference["memory_gib"]
    })

workloads = pd.DataFrame(workload_rows)
workloads.to_csv(data_folder / "workload_profiles.csv", index=False)
display(workloads)

base_latency = np.average(workloads["rpc_ms"], weights=workloads["relative_demand"])
mean_cpu = workloads["mean_cpu"].sum()
peak_cpu = workloads["peak_cpu"].sum()
peak_memory = workloads["peak_memory_gib"].sum()

selected_sizes = ["standard-1", "standard-2", "standard-3", "standard-4"]
instance_types = sizes.loc[selected_sizes]
rates = pricing.set_index("parameter")["rate"]
locations = list(regions)
seconds_per_month = 30 * 24 * 60 * 60

# Planning constraints (adjust if needed, not observed company limits).
budget_limit = 5000
latency_limit = 500
rows = []

for region_count in range(1, len(locations) + 1):
    for active_regions in combinations(locations, region_count):
        nearest = latency_matrix[list(active_regions)].idxmin(axis=1)
        network_latency = latency_matrix[list(active_regions)].min(axis=1).mean()
        shares = nearest.value_counts().reindex(active_regions, fill_value=0) / len(locations)

        for tier, size in enumerate(selected_sizes, start=1):
            instance = instance_types.loc[size]
            cpu = instance["vcpu"]
            memory = instance["memory_gib"]
            disk = instance["disk_gb"]

            max_cpu_demand = peak_cpu * shares.max()
            cpu_ok = max_cpu_demand <= cpu
            memory_ok = peak_memory <= memory

            # CPU is charged for active usage; memory and disk are provisioned.
            monthly_cost = seconds_per_month * (
                mean_cpu * rates["cpu_vcpu_second"]
                + region_count * (memory * rates["memory_gib_second"] + disk * rates["disk_gb_second"])
            )
            predicted_latency = base_latency + network_latency
            eligible = set(active_regions).issubset(locations)
            within_budget = monthly_cost <= budget_limit
            within_latency = predicted_latency <= latency_limit
            feasible = eligible and cpu_ok and memory_ok and within_budget and within_latency

            rows.append({
                "active_regions": ", ".join(active_regions),
                "region_count": region_count,
                "tier_index": tier,
                "instance_type": size,
                "vcpu_per_region": cpu,
                "memory_gib_per_region": memory,
                "peak_cpu_demand": max_cpu_demand,
                "peak_memory_demand_gib": peak_memory,
                "routing": "; ".join(f"{source}: {target}" for source, target in nearest.items()),
                "cost_usd_month": monthly_cost,
                "latency_ms": predicted_latency,
                "cpu_ok": cpu_ok,
                "memory_ok": memory_ok,
                "budget_ok": within_budget,
                "latency_ok": within_latency,
                "eligible": eligible,
                "feasible": feasible
            })

all_df = pd.DataFrame(rows)
all_df.to_csv(data_folder / "all_scenarios.csv", index=False)

# Only feasible plans can be on the frontier.
feasible_df = all_df[all_df["feasible"]].copy()
feasible_df.to_csv(data_folder / "feasible_scenarios.csv", index=False)
df = pd.read_csv(data_folder / "feasible_scenarios.csv")

assert len(all_df) == 124
if df.empty:
    display(all_df[["cpu_ok", "memory_ok", "budget_ok", "latency_ok", "eligible"]].sum())
    raise RuntimeError("No feasible plans. Review the workload profiles and planning limits.")

print("Deployment configurations created!")
print("Shape:", df.shape)
print("Total configurations:", len(all_df))
print("Feasible configurations:", len(df))
print("Excluded configurations:", len(all_df) - len(df))
display(df.head())

,service,observations,relative_demand,rpc_ms,mean_cpu,peak_cpu,peak_memory_gib
0,MS_10489,2,0.000010,18.044185,0.023911,0.025333,0.781114
1,MS_19988,1,0.000004,25.286396,0.029418,0.029418,0.629443
2,MS_29860,1,0.000043,15.055216,0.088853,0.088853,0.694230
3,MS_43157,2,0.000045,3.173659,0.089044,0.091485,0.663337
4,MS_48534,2,0.000004,27.796273,0.028748,0.029160,0.644674
5,MS_491,1,0.000010,18.139073,0.039609,0.039609,0.790181


Deployment configurations created!
Shape: (93, 17)
Total configurations: 124
Feasible configurations: 93
Excluded configurations: 31


,active_regions,region_count,tier_index,instance_type,vcpu_per_region,memory_gib_per_region,peak_cpu_demand,peak_memory_demand_gib,routing,cost_usd_month,latency_ms,cpu_ok,memory_ok,budget_ok,latency_ok,eligible,feasible
0,US East,1,2,standard-2,1.0,6.0,0.303858,4.202978,US East: US East; US West: US East; London: US...,56.587625,116.145075,True,True,True,True,True,True
1,US East,1,3,standard-3,2.0,8.0,0.303858,4.202978,US East: US East; US West: US East; London: US...,70.273385,116.145075,True,True,True,True,True,True
2,US East,1,4,standard-4,4.0,12.0,0.303858,4.202978,US East: US East; US West: US East; London: US...,96.919145,116.145075,True,True,True,True,True,True
3,US West,1,2,standard-2,1.0,6.0,0.303858,4.202978,US East: US West; US West: US West; London: US...,56.587625,112.685585,True,True,True,True,True,True
4,US West,1,3,standard-3,2.0,8.0,0.303858,4.202978,US East: US West; US West: US West; London: US...,70.273385,112.685585,True,True,True,True,True,True


## Section 2: Data Exploration

## Section 3: Pareto Frontier Calculation

## Section 4: Strategic Analysis

## Section 5: Visualization